# Who loses in a stock market crash? Four kinds of households

The stock-crash model of `stock_crash_model.ipynb`, with its one household split into four, the groups the Federal Reserve follows by
wealth: the bottom 50%, the next 40%, the next 9% and the top 1%. They differ in how many live from paycheck to paycheck, how much of
the stock market they own, and how much their jobs move with the economy. Every function here is explained in
`doc/lisp_macro_models_manual.md` (the section *Four kinds of households*), which also says what the model leaves out. Responses are by
the quarter: quarter 1 is the quarter of the shock.

## 1. Who owns what: the Fed's data

`wealth-by-type` fetches, from FRED, the Federal Reserve's Distributional Financial Accounts: the share of the nation's net worth, and
of its corporate equities and mutual fund shares, held by each group.

In [ ]:
(define wealth (wealth-by-type creds))
(display-table wealth)

## 2. A 20% fall in the stock market

`types-model` runs Dynare. `types-crash-losses` says, for each group, how much its income and its spending fall (percent from the group's
own normal).

In [ ]:
(define base (types-model :wealth wealth))
(display-table (types-crash-losses base :fall 20))

In [ ]:
(plot-types-crash base :fall 20)

## 3. Does it matter that the households differ?

With nobody living from paycheck to paycheck, and every group's hours moving as the economy's, the four groups' spending adds up to the
one-household crash model's. Compare the economy under the two.

In [ ]:
(define same (types-model :parameters '((hand_to_mouth_bottom50 . 0) (hand_to_mouth_next40 . 0) (hand_to_mouth_next9 . 0)
                                       (hand_to_mouth_top1 . 0)
                                       (earnings_cyclicality_bottom50 . 1) (earnings_cyclicality_next40 . 1)
                                       (earnings_cyclicality_next9 . 1) (earnings_cyclicality_top1 . 1))))
(display-table (table-head (types-crash-responses same :fall 20) 6))
(display-table (table-head (types-crash-responses base :fall 20) 6))

## 4. What if the middle owns more of the stock market than the Fed's table says?

Much of the middle's stock is in retirement accounts, which that table may not fully count. Give the next 40% a quarter of the market
(the four shares must add to 1).

In [ ]:
(define middle-owns-more
  (types-model :parameters '((stock_share_bottom50 . 0.006) (stock_share_next40 . 0.25) (stock_share_next9 . 0.30) (stock_share_top1 . 0.444))))
(display-table (types-crash-losses middle-owns-more :fall 20))

## 5. What if the bottom half could plan ahead?

`hand_to_mouth_bottom50` is the part of the bottom half that spends all it earns each quarter. Set it to 0.

In [ ]:
(define planners (types-model :parameters '((hand_to_mouth_bottom50 . 0))))
(display-table (types-crash-losses planners :fall 20))

## 6. Other kinds of surprise

The same model, any of the crash model's six surprises, by group. A typical surprise to the Fed's rate:

In [ ]:
(display-table (types-model-responses base "fed_shock"))